In [ ]:
# ---------------------------------------------------------
# Environment Setup
# ---------------------------------------------------------
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# ---------------------------------------------------------
# Messages (Bank Conversation)
# ---------------------------------------------------------
from langchain_core.messages import AIMessage, HumanMessage

messages = [
    AIMessage(content="Welcome to the bank, how can I assist you?", name="BankBot"),
    HumanMessage(content="I want to open an account", name="Customer"),
    AIMessage(content="Would you like a Savings or Current account?", name="BankBot"),
    HumanMessage(content="I want a Savings account", name="Customer"),
]

for message in messages:
    message.pretty_print()

# ---------------------------------------------------------
# LLM Setup
# ---------------------------------------------------------
from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen-qwq-32b")
result = llm.invoke(messages)
print(result.response_metadata)

# ---------------------------------------------------------
# Tool Example (Bank Utility)
# ---------------------------------------------------------
def calculate_interest(balance: int, rate: float) -> float:
    """Calculate simple interest"""
    return balance * rate / 100

llm_with_tools = llm.bind_tools([calculate_interest])

tool_call = llm_with_tools.invoke(
    [HumanMessage(content="What is interest on 1000 at 5%?", name="Customer")]
)
print(tool_call.tool_calls)

# ---------------------------------------------------------
# State Schema
# ---------------------------------------------------------
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from typing import Annotated
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# Initial messages
initial_messages = [
    AIMessage(content="Welcome to the bank, how can I assist you?", name="BankBot"),
    HumanMessage(content="I want to open an account", name="Customer"),
]

ai_message = AIMessage(content="Would you like a Savings or Current account?", name="BankBot")
add_messages(initial_messages, ai_message)

# ---------------------------------------------------------
# Node Functionality
# ---------------------------------------------------------
def bank_llm_tool(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# ---------------------------------------------------------
# Graph Construction
# ---------------------------------------------------------
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

builder = StateGraph(State)
builder.add_node("bank_llm_tool", bank_llm_tool)
builder.add_edge(START, "bank_llm_tool")
builder.add_edge("bank_llm_tool", END)

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

# ---------------------------------------------------------
# Graph Invocation
# ---------------------------------------------------------
messages = graph.invoke({"messages": "What is interest on 1000 at 5%"})
for message in messages["messages"]:
    message.pretty_print()

# ---------------------------------------------------------
# ToolNode Integration
# ---------------------------------------------------------
from langgraph.prebuilt import ToolNode, tools_condition

tools = [calculate_interest]
builder = StateGraph(State)

builder.add_node("bank_llm_tool", bank_llm_tool)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "bank_llm_tool")
builder.add_conditional_edges("bank_llm_tool", tools_condition)
builder.add_edge("tools", END)

graph_builder = builder.compile()
display(Image(graph_builder.get_graph().draw_mermaid_png()))

# ---------------------------------------------------------
# Final Invocations
# ---------------------------------------------------------
messages = graph_builder.invoke({"messages": "What is interest on 1000 at 5%"})
for message in messages["messages"]:
    message.pretty_print()

messages = graph_builder.invoke({"messages": "Tell me about Current accounts"})
for message in messages["messages"]:
    message.pretty_print()
